# 03 – Fra piksler til prediksjoner

**Tid:** 75 minutter. **Mål:** følge formene fra bilder til klasseskårer,
forstå vektede summer og bygge en lineær modell. Modellen trenes ikke ennå.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import download_mnist

treningsdata, testdata = download_mnist()
print(f"Treningsbilder: {len(treningsdata)}, testbilder: {len(testdata)}")


## Eksempel: en vektet sum

En lineær modell multipliserer hvert inndatatall med en vekt, summerer og
legger til et konstantledd (*bias*). Vektene bestemmer påvirkningen fra hver verdi.
I uttrykket `x @ W + b` er `@` matrisemultiplikasjon, ikke elementvis gange.


In [ ]:
# To målinger gir én skår. Tallene er valgt for hånd.
x = torch.tensor([[2.0, 3.0]])
vekter = torch.tensor([[0.5], [-1.0]])
konstantledd = torch.tensor([0.2])
skaar = x @ vekter + konstantledd
print("Skår:", skaar, "Form:", skaar.shape)
print("Samme beregning for hånd:", 2 * 0.5 + 3 * (-1) + 0.2)


## Oppgave 1: Beregn to skårer per eksempel

Bruk `innganger = torch.tensor([[1., 2., 3.], [4., 5., 6.]])`,
`vekter = torch.tensor([[1., 0.], [0., 1.], [1., -1.]])` og
`konstantledd = torch.tensor([0.5, -0.5])`.
Beregn `skaarer`.
Forutsi formen før du kjører koden, og skriv så ut resultatet.

<details>
<summary>Hint</summary>

En `[2, 3]`-matrise ganget med en `[3, 2]`-matrise gir formen `[2, 2]`.

</details>


In [ ]:
# Skriv din løsning her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert torch.allclose(skaarer, torch.tensor([[4.5, -1.5], [10.5, -1.5]]))


### Sjekk forståelsen

Hva representerer radene og kolonnene i resultatet? Hvorfor må de to indre
dimensjonene stemme ved matrisemultiplikasjon? Hvordan ville `*` oppført seg her?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

...


## Eksempel: flat ut bilder, behold bildeskillet

En lineær modell trenger én rekke tall per bilde. `flatten(start_dim=1)`
beholder dimensjon 0 og slår sammen resten. Piksler fjernes ikke, men den
todimensjonale plasseringen er ikke lenger eksplisitt i formen.


In [ ]:
smaabilder = torch.arange(24, dtype=torch.float32).reshape(2, 1, 3, 4)
flate = smaabilder.flatten(start_dim=1)
print(smaabilder.shape, "→", flate.shape)


## Nå trenger vi grupper av bilder

Til nå har vi hentet enkeltbilder med `treningsdata[indeks]`. Modellen
kan regne på flere bilder samtidig, og under trening vil vi oppdatere
den etter hver lille gruppe. En slik gruppe kalles en **batch**.

`DataLoader` samler bilder og etiketter fra datasettet. Oppsettet er
synlig under: `batch_size=64` gir opptil 64 bilder, og `shuffle=True`
stokker rekkefølgen for hver gjennomgang. Den siste gruppen kan være mindre.
Datasettet inneholder eksemplene; lasteren bestemmer hvordan de grupperes.
`next(iter(...))` henter én gruppe som vi kan undersøke.


In [ ]:
from torch.utils.data import DataLoader

treningslaster = DataLoader(treningsdata, batch_size=64, shuffle=True)
bilder, etiketter = next(iter(treningslaster))
print("Bilder:", bilder.shape, bilder.dtype)
print("Etiketter:", etiketter.shape, etiketter.dtype)


### Sjekk forståelsen

Forklar hvert tall i bildegruppens form. Hvorfor har etikettene bare
én dimensjon? Hva ville endret seg om `batch_size` var 16?

Skriv svaret ditt i Markdown-cellen under.


**Mitt svar:**

...


## Oppgave 2: Lag ti skårer for MNIST

Hent `bilder, etiketter` fra `treningslaster`. Lag `flate_bilder` med én rad
per bilde. Lag tilfeldige `mnistvekter` med formen `[784, 10]`, gang dem med
0.01, og lag `mnistkonstantledd` som ti nuller. Beregn `mnistskaarer`.
Bruk `argmax(dim=1)` til å lage `prediksjoner`. Skriv ut alle formene.

<details>
<summary>Hint</summary>

`torch.randn` lager tilfeldige tall. `argmax(dim=1)` finner kolonnen med høyest skår i hver rad.

</details>


In [ ]:
# Skriv din løsning her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert flate_bilder.shape == (bilder.shape[0], 784)
assert mnistskaarer.shape == (bilder.shape[0], 10)
assert prediksjoner.shape == etiketter.shape


## Eksempel: la `nn.Linear` håndtere parametrene

`nn.Linear(3, 2)` lærer tre vekter og ett konstantledd per utgang.
Den lagrer vektene som `[utganger, innganger]`, motsatt av vår håndlagde
matrise. Laget utfører den nødvendige transponeringen selv.
`nn.Sequential` kjører flere lag etter hverandre.


In [ ]:
lite_lag = nn.Linear(3, 2)
print("Resultat:", lite_lag(torch.ones(4, 3)).shape)
print("Vekter:", lite_lag.weight.shape, "Konstantledd:", lite_lag.bias.shape)


## Oppgave 3: Bygg en lineær bildemodell

Lag `modell = nn.Sequential(...)` med `nn.Flatten()` og et passende `nn.Linear`.
Modellen skal motta bildene direkte og gi ti skårer per bilde.
Beregn `modellens_skaarer`, og tell parametrene med
`sum(parameter.numel() for parameter in modell.parameters())`.

<details>
<summary>Hint</summary>

Hvert bilde har `28 * 28 = 784` piksler. Det er ti mulige sifre.

</details>


In [ ]:
# Skriv din løsning her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert modellens_skaarer.shape == (len(bilder), 10)
assert sum(p.numel() for p in modell.parameters()) == 7850


### Sjekk forståelsen

Skriv formene gjennom modellen for en gruppe på 32 MNIST-bilder.
Hvorfor har modellen 7 850 parametere? Hvorfor forventer vi ikke gode
prediksjoner før trening? Er de ti skårene sannsynligheter?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

...
